# KPT Signal Fusion — Notebook

**Problem:** The Food-Order-Ready (FOR) signal used for Kitchen Preparation Time (KPT) estimation is frequently reactive — merchants press the button *after* the rider arrives, not when food is genuinely ready.  This inflates KPT estimates, forces riders to wait, and degrades delivery ETA accuracy.

**Approach:** Multi-signal fusion (POS completion + pickup scan + debiased FOR) with per-merchant bias calibration and confidence-weighted prediction.

---
**Notebook structure**
1. Environment setup
2. Data generation & validation
3. Signal enrichment & bias analysis
4. KPT fusion — model fitting and inference
5. Dispatch simulation
6. Quantitative evaluation
7. Ablation study
8. Segment analysis
9. Visualisations
10. Results summary

In [ ]:
# ── Environment setup ─────────────────────────────────────────────────────────
import sys
import warnings
import logging
from pathlib import Path

warnings.filterwarnings('ignore')
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(levelname)-8s  %(name)s  %(message)s',
    datefmt='%H:%M:%S',
)

# Make the src package importable from the notebooks/ directory
ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__, '  Pandas:', pd.__version__)

## 1. Data generation & validation

In [ ]:
from src.config import N_MERCHANTS, N_ORDERS, SIM_DAYS, RANDOM_SEED
from src.data_generation import generate_synthetic_merchants, generate_synthetic_orders
from src.pipeline import validate_dataset

merchants = generate_synthetic_merchants(n=N_MERCHANTS, seed=RANDOM_SEED)
orders = generate_synthetic_orders(
    merchants=merchants, n_orders=N_ORDERS, sim_days=SIM_DAYS, seed=RANDOM_SEED
)
validate_dataset(orders, merchants)

print(f'Merchants: {len(merchants):,}  |  Orders: {len(orders):,}')
print(f'True FOR bias rate: {orders["is_merchant_biased"].mean():.1%}')
display(orders.describe().round(2))

## 2. Signal enrichment & bias analysis

In [ ]:
from src.signal_enrichment import enrich_with_signals

enriched = enrich_with_signals(orders)

# Bias detection quality
from sklearn.metrics import precision_recall_fscore_support  # type: ignore
try:
    p, r, f1, _ = precision_recall_fscore_support(
        enriched['is_merchant_biased'],
        enriched['is_for_biased_flag'],
        average='binary',
        zero_division=0,
    )
    print(f'Bias detection  P={p:.2f}  R={r:.2f}  F1={f1:.2f}')
except ImportError:
    tp = (enriched['is_for_biased_flag'] & enriched['is_merchant_biased']).sum()
    fp = (enriched['is_for_biased_flag'] & ~enriched['is_merchant_biased']).sum()
    fn = (~enriched['is_for_biased_flag'] & enriched['is_merchant_biased']).sum()
    p  = tp / (tp + fp) if (tp + fp) else 0
    r  = tp / (tp + fn) if (tp + fn) else 0
    f1 = 2*p*r/(p+r) if (p+r) else 0
    print(f'Bias detection  P={p:.2f}  R={r:.2f}  F1={f1:.2f}')

print(f'\nSignal availability:')
print(f'  POS:  {enriched["has_pos"].mean():.1%}')
print(f'  Scan: {enriched["has_scan"].mean():.1%}')
print(f'  Mean signal confidence: {enriched["signal_confidence"].mean():.3f}')

## 3. KPT Fusion — model fitting and inference

In [ ]:
from src.fusion_engine import KPTSignalFusion

model = KPTSignalFusion()
model.compute_bias_offsets(enriched)
estimated = model.fuse_signals(enriched)

print(f'Mean predicted KPT:  {estimated["predicted_kpt_min"].mean():.2f} min')
print(f'Mean true KPT:       {estimated["true_kpt_min"].mean():.2f} min')
print(f'Mean raw FOR:        {estimated["reported_FOR_delta_min"].mean():.2f} min')
print()
print('Sample bias offsets (top 5 by magnitude):')
display(
    model.bias_offsets_
    .reindex(model.bias_offsets_['bias_offset_min'].abs().nlargest(5).index)
    .reset_index(drop=True)
)

## 4. Dispatch simulation

In [ ]:
from src.simulation import simulate_dispatch

simulated = simulate_dispatch(estimated, pred_col='predicted_kpt_min')
baseline_sim = simulate_dispatch(estimated, pred_col='reported_FOR_delta_min')

print(f'Avg rider wait — Baseline: {baseline_sim["rider_wait_min"].mean():.3f} min')
print(f'Avg rider wait — Proposed: {simulated["rider_wait_min"].mean():.3f} min')
print(f'Delay rate     — Baseline: {baseline_sim["delay_flag"].mean():.1%}')
print(f'Delay rate     — Proposed: {simulated["delay_flag"].mean():.1%}')

## 5. Quantitative evaluation

In [ ]:
from src.evaluation import evaluate_metrics

bm = evaluate_metrics(baseline_sim, 'reported_FOR_delta_min')
pm = evaluate_metrics(simulated, 'predicted_kpt_min')

metrics_df = pd.DataFrame({'Baseline': bm, 'Proposed': pm}).T
metrics_df['Δ (%)'] = (
    (metrics_df.loc['Baseline'] - metrics_df.loc['Proposed'])
    / metrics_df.loc['Baseline'] * 100
).round(1)
display(metrics_df.round(3))

## 6. Ablation study

In [ ]:
from src.evaluation import run_ablation_study

ablation = run_ablation_study(estimated, model)
display(ablation[['MAE', 'RMSE', 'P90_error', 'within_2min_pct', 'avg_rider_wait_min']].round(3))

## 7. Segment analysis

Testing whether improvements hold across key operational sub-populations.

In [ ]:
from src.evaluation import segment_experiment

segments = {
    'Rush hours': simulated['is_rush_hour'],
    'Off-peak': ~simulated['is_rush_hour'],
    'Biased merchants': simulated['is_for_biased_flag'],
    'POS available': simulated['has_pos'],
    'POS unavailable': ~simulated['has_pos'],
    'High confidence (≥0.5)': simulated['signal_confidence_score'] >= 0.5,
    'Low confidence (<0.3)': simulated['signal_confidence_score'] < 0.3,
}

seg_results = {}
for name, mask in segments.items():
    result = segment_experiment(simulated, name, mask)
    if result is not None:
        seg_results[name] = result

seg_df = pd.DataFrame(seg_results).T
display(seg_df.round(3))

## 8. Visualisations

In [ ]:
from src.plotting import plot_error_distributions, plot_wait_time_comparison, plot_ablation
from pathlib import Path

FIG_DIR = Path('figures')
FIG_DIR.mkdir(exist_ok=True)

fig1 = plot_error_distributions(simulated, save_path=FIG_DIR / 'kpt_evaluation_overview.png')
plt.show()

fig2 = plot_wait_time_comparison(seg_results, save_path=FIG_DIR / 'segment_improvement.png')
plt.show()

fig3 = plot_ablation(ablation, save_path=FIG_DIR / 'ablation_study.png')
plt.show()

## 9. Results summary

In [ ]:
from src.pipeline import print_results_summary

pipeline = {
    'base_metrics': bm,
    'prop_metrics': pm,
}
print_results_summary(pipeline)